In [3]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5

TARGET = "addicted_label"
ID_COL = "id"

print("All the imports are successfull")



All the imports are successfull


In [5]:
train_df = pd.read_csv("datasets/train.csv")
test_df = pd.read_csv("datasets/test.csv")

print(f"Train dataset shape : {train_df.shape}")
print(f"Test dataset shape : {test_df.shape}")
print(train_df[TARGET].value_counts(normalize=True).sort_index())

Train dataset shape : (691369, 14)
Test dataset shape : (296302, 13)
addicted_label
0    0.290576
1    0.709424
Name: proportion, dtype: float64


In [6]:
def create_features(df):

    df = df.copy()

    eps = 1e-6

    # Combined known usage
    df["known_usage_hours"] = (
        df["social_media_hours"]
        + df["gaming_hours"]
        + df["work_study_hours"]
    )

    # Difference between reported total screen time
    # and the listed usage categories
    df["unaccounted_screen_time"] = (
        df["daily_screen_time_hours"]
        - df["known_usage_hours"]
    )

    # Usage-to-sleep relationship
    df["screen_sleep_ratio"] = (
        df["daily_screen_time_hours"]
        / (df["sleep_hours"] + eps)
    )

    # Usage composition
    df["social_media_ratio"] = (
        df["social_media_hours"]
        / (
            df["daily_screen_time_hours"]
            + eps
        )
    )

    df["gaming_ratio"] = (
        df["gaming_hours"]
        / (
            df["daily_screen_time_hours"]
            + eps
        )
    )

    df["work_study_ratio"] = (
        df["work_study_hours"]
        / (
            df["daily_screen_time_hours"]
            + eps
        )
    )

    # Weekend behavior
    df["weekend_screen_difference"] = (
        df["weekend_screen_time"]
        - df["daily_screen_time_hours"]
    )

    df["weekend_screen_ratio"] = (
        df["weekend_screen_time"]
        / (
            df["daily_screen_time_hours"]
            + eps
        )
    )

    # Engagement intensity
    df["notifications_per_screen_hour"] = (
        df["notifications_per_day"]
        / (
            df["daily_screen_time_hours"]
            + eps
        )
    )

    df["app_opens_per_screen_hour"] = (
        df["app_opens_per_day"]
        / (
            df["daily_screen_time_hours"]
            + eps
        )
    )

    df["notifications_per_app_open"] = (
        df["notifications_per_day"]
        / (
            df["app_opens_per_day"]
            + eps
        )
    )

    # Sleep-related features
    df["sleep_deficit_8h"] = (
        8
        - df["sleep_hours"]
    )

    df["short_sleep_flag"] = (
        df["sleep_hours"] < 7
    ).astype(int)

    # High-use indicators
    df["high_screen_time_flag"] = (
        df["daily_screen_time_hours"] >= 8
    ).astype(int)

    df["high_notification_flag"] = (
        df["notifications_per_day"] >= 100
    ).astype(int)

    # Useful interactions
    df["screen_time_x_social_media"] = (
        df["daily_screen_time_hours"]
        * df["social_media_hours"]
    )

    df["screen_time_x_app_opens"] = (
        df["daily_screen_time_hours"]
        * df["app_opens_per_day"]
    )

    return df

In [7]:
train_features_df = create_features(train_df)
test_features_df = create_features(test_df)

print(
    "Train after feature engineering:",
    train_features_df.shape
)

print(
    "Test after feature engineering:",
    test_features_df.shape
)

Train after feature engineering: (691369, 31)
Test after feature engineering: (296302, 30)


In [8]:
X = train_features_df.drop(
    columns=[TARGET, ID_COL],
    errors="ignore"
)

y = train_features_df[TARGET].astype(int)

X_test = test_features_df.drop(
    columns=[ID_COL],
    errors="ignore"
)

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

print("\nFeature columns:")
print(X.columns.tolist())

X shape: (691369, 29)
X_test shape: (296302, 29)

Feature columns:
['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'known_usage_hours', 'unaccounted_screen_time', 'screen_sleep_ratio', 'social_media_ratio', 'gaming_ratio', 'work_study_ratio', 'weekend_screen_difference', 'weekend_screen_ratio', 'notifications_per_screen_hour', 'app_opens_per_screen_hour', 'notifications_per_app_open', 'sleep_deficit_8h', 'short_sleep_flag', 'high_screen_time_flag', 'high_notification_flag', 'screen_time_x_social_media', 'screen_time_x_app_opens']


In [9]:
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_columns = [
    col for col in X.columns
    if col not in categorical_columns
]

print("Categorical columns:")
print(categorical_columns)

print("\nNumeric columns:")
print(numeric_columns)

Categorical columns:
['gender', 'stress_level', 'academic_work_impact']

Numeric columns:
['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'known_usage_hours', 'unaccounted_screen_time', 'screen_sleep_ratio', 'social_media_ratio', 'gaming_ratio', 'work_study_ratio', 'weekend_screen_difference', 'weekend_screen_ratio', 'notifications_per_screen_hour', 'app_opens_per_screen_hour', 'notifications_per_app_open', 'sleep_deficit_8h', 'short_sleep_flag', 'high_screen_time_flag', 'high_notification_flag', 'screen_time_x_social_media', 'screen_time_x_app_opens']


In [10]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(
    "Using 5-fold StratifiedKFold"
)

Using 5-fold StratifiedKFold


# Logistic Regression

In [12]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                C=1.0,
                random_state=RANDOM_STATE
            )
        )
    ]
)

logistic_oof = np.zeros(
    len(X)
)

logistic_fold_scores = []

for fold, (
    train_index,
    valid_index
) in enumerate(
    cv.split(X, y),
    start=1
):

    X_train_fold = X.iloc[
        train_index
    ]

    y_train_fold = y.iloc[
        train_index
    ]

    X_valid_fold = X.iloc[
        valid_index
    ]

    y_valid_fold = y.iloc[
        valid_index
    ]

    logistic_model.fit(
        X_train_fold,
        y_train_fold
    )

    valid_probability = (
        logistic_model
        .predict_proba(
            X_valid_fold
        )[:, 1]
    )

    logistic_oof[
        valid_index
    ] = valid_probability

    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_probability
    )

    logistic_fold_scores.append(
        fold_auc
    )

    print(
        f"Fold {fold}: "
        f"AUC = {fold_auc:.6f}"
    )

logistic_auc = roc_auc_score(
    y,
    logistic_oof
)

print(
    f"Logistic OOF AUC: "
    f"{logistic_auc:.6f}"
)
print(
    f"Mean fold AUC: "
    f"{np.mean(logistic_fold_scores):.6f}"
)
print(
    f"Fold std: "
    f"{np.std(logistic_fold_scores):.6f}"
)


Fold 1: AUC = 0.922439
Fold 2: AUC = 0.922847
Fold 3: AUC = 0.923683
Fold 4: AUC = 0.924038
Fold 5: AUC = 0.923248
Logistic OOF AUC: 0.923249
Mean fold AUC: 0.923251
Fold std: 0.000571


# CatBoost inline

In [13]:
X_catboost = X.copy()
X_test_catboost = X_test.copy()

for col in categorical_columns:

    X_catboost[col] = (
        X_catboost[col]
        .fillna("MISSING")
        .astype(str)
    )

    X_test_catboost[col] = (
        X_test_catboost[col]
        .fillna("MISSING")
        .astype(str)
    )

catboost_oof = np.zeros(
    len(X)
)

catboost_test_predictions = np.zeros(
    len(X_test)
)

catboost_fold_scores = []

for fold, (
    train_index,
    valid_index
) in enumerate(
    cv.split(
        X_catboost,
        y
    ),
    start=1
):

    X_train_fold = (
        X_catboost
        .iloc[train_index]
    )

    y_train_fold = (
        y.iloc[train_index]
    )

    X_valid_fold = (
        X_catboost
        .iloc[valid_index]
    )

    y_valid_fold = (
        y.iloc[valid_index]
    )

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=(
            RANDOM_STATE
            + fold
        ),
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=(
            categorical_columns
        ),
        eval_set=(
            X_valid_fold,
            y_valid_fold
        ),
        early_stopping_rounds=100,
        verbose=False
    )

    valid_probability = (
        model.predict_proba(
            X_valid_fold
        )[:, 1]
    )

    catboost_oof[
        valid_index
    ] = valid_probability

    test_probability = (
        model.predict_proba(
            X_test_catboost
        )[:, 1]
    )

    catboost_test_predictions += (
        test_probability
        / N_SPLITS
    )

    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_probability
    )

    catboost_fold_scores.append(
        fold_auc
    )

    print(
        f"Fold {fold}: "
        f"AUC = {fold_auc:.6f}"
    )

catboost_auc = roc_auc_score(
    y,
    catboost_oof
)

print(
    f"CatBoost OOF AUC: "
    f"{catboost_auc:.6f}"
)
print(
    f"Mean fold AUC: "
    f"{np.mean(catboost_fold_scores):.6f}"
)
print(
    f"Fold std: "
    f"{np.std(catboost_fold_scores):.6f}"
)

Fold 1: AUC = 0.954168
Fold 2: AUC = 0.954860
Fold 3: AUC = 0.955165
Fold 4: AUC = 0.956095
Fold 5: AUC = 0.954778
CatBoost OOF AUC: 0.955011
Mean fold AUC: 0.955013
Fold std: 0.000630


### Comparing the models

In [14]:
results = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "CatBoost"
    ],
    "OOF_AUC": [
        logistic_auc,
        catboost_auc
    ],
    "mean_fold_AUC": [
        np.mean(
            logistic_fold_scores
        ),
        np.mean(
            catboost_fold_scores
        )
    ],
    "fold_std": [
        np.std(
            logistic_fold_scores
        ),
        np.std(
            catboost_fold_scores
        )
    ]
})

results = results.sort_values(
    "OOF_AUC",
    ascending=False
)

display(results)

,model,OOF_AUC,mean_fold_AUC,fold_std
1,CatBoost,0.955011,0.955013,0.000630
0,Logistic Regression,0.923249,0.923251,0.000571


In [15]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Make fresh copies
X_ml = X.copy()
X_test_ml = X_test.copy()

# Numeric preprocessing
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ]
)

# Combined preprocessing
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_columns
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_columns
        )
    ],
    remainder="drop"
)

# Fit only on training features
X_encoded = tree_preprocessor.fit_transform(X_ml)

# Transform test features
X_test_encoded = tree_preprocessor.transform(
    X_test_ml
)

print("Encoded train shape:", X_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)

Encoded train shape: (691369, 29)
Encoded test shape: (296302, 29)


In [16]:
def train_with_cv(
    model_builder,
    X_data,
    y_data,
    X_test_data,
    model_name
):

    oof_predictions = np.zeros(
        len(X_data)
    )

    test_predictions = np.zeros(
        len(X_test_data)
    )

    fold_scores = []

    for fold, (
        train_index,
        valid_index
    ) in enumerate(
        cv.split(
            X_data,
            y_data
        ),
        start=1
    ):

        # Split training data
        if isinstance(
            X_data,
            pd.DataFrame
        ):

            X_train_fold = (
                X_data.iloc[
                    train_index
                ]
            )

            X_valid_fold = (
                X_data.iloc[
                    valid_index
                ]
            )

        else:

            X_train_fold = (
                X_data[
                    train_index
                ]
            )

            X_valid_fold = (
                X_data[
                    valid_index
                ]
            )

        y_train_fold = (
            y_data.iloc[
                train_index
            ]
        )

        y_valid_fold = (
            y_data.iloc[
                valid_index
            ]
        )

        # Create a fresh model
        model = model_builder(
            fold
        )

        # Train
        model.fit(
            X_train_fold,
            y_train_fold
        )

        # Validation probabilities
        valid_predictions = (
            model.predict_proba(
                X_valid_fold
            )[:, 1]
        )

        oof_predictions[
            valid_index
        ] = valid_predictions

        # Test probabilities
        fold_test_predictions = (
            model.predict_proba(
                X_test_data
            )[:, 1]
        )

        test_predictions += (
            fold_test_predictions
            / N_SPLITS
        )

        # ROC-AUC
        fold_auc = roc_auc_score(
            y_valid_fold,
            valid_predictions
        )

        fold_scores.append(
            fold_auc
        )

        print(
            f"{model_name} | "
            f"Fold {fold} | "
            f"AUC: {fold_auc:.6f}"
        )

    overall_auc = roc_auc_score(
        y_data,
        oof_predictions
    )

    print("\n" + "=" * 70)

    print(
        f"{model_name} "
        f"OOF ROC-AUC: "
        f"{overall_auc:.6f}"
    )

    print(
        f"Mean Fold AUC: "
        f"{np.mean(fold_scores):.6f}"
    )

    print(
        f"Fold Standard Deviation: "
        f"{np.std(fold_scores):.6f}"
    )

    print("=" * 70)

    return {
        "name": model_name,
        "oof": oof_predictions,
        "test": test_predictions,
        "auc": overall_auc,
        "fold_scores": fold_scores
    }

In [17]:
from sklearn.ensemble import RandomForestClassifier

def build_random_forest(fold):

    return RandomForestClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=(
            RANDOM_STATE + fold
        ),
        n_jobs=-1
    )


random_forest_result = train_with_cv(
    model_builder=build_random_forest,
    X_data=X_encoded,
    y_data=y,
    X_test_data=X_test_encoded,
    model_name="Random Forest"
)

Random Forest | Fold 1 | AUC: 0.941596
Random Forest | Fold 2 | AUC: 0.942054
Random Forest | Fold 3 | AUC: 0.943051
Random Forest | Fold 4 | AUC: 0.943949
Random Forest | Fold 5 | AUC: 0.942843

Random Forest OOF ROC-AUC: 0.942694
Mean Fold AUC: 0.942699
Fold Standard Deviation: 0.000817


In [18]:
from sklearn.ensemble import (
    HistGradientBoostingClassifier
)

def build_hist_gradient_boosting(
    fold
):

    return HistGradientBoostingClassifier(
        learning_rate=0.04,
        max_iter=600,
        max_leaf_nodes=31,
        min_samples_leaf=15,
        l2_regularization=1.0,
        random_state=(
            RANDOM_STATE + fold
        )
    )


hist_gradient_result = train_with_cv(
    model_builder=(
        build_hist_gradient_boosting
    ),
    X_data=X_encoded,
    y_data=y,
    X_test_data=X_test_encoded,
    model_name=(
        "HistGradientBoosting"
    )
)

HistGradientBoosting | Fold 1 | AUC: 0.960728
HistGradientBoosting | Fold 2 | AUC: 0.961242
HistGradientBoosting | Fold 3 | AUC: 0.961779
HistGradientBoosting | Fold 4 | AUC: 0.962426
HistGradientBoosting | Fold 5 | AUC: 0.961519

HistGradientBoosting OOF ROC-AUC: 0.961537
Mean Fold AUC: 0.961539
Fold Standard Deviation: 0.000564


In [19]:
def build_lightgbm(fold):

    return LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=1.0,
        random_state=(
            RANDOM_STATE + fold
        ),
        n_jobs=-1,
        verbosity=-1
    )


lightgbm_result = train_with_cv(
    model_builder=build_lightgbm,
    X_data=X_encoded,
    y_data=y,
    X_test_data=X_test_encoded,
    model_name="LightGBM"
)

LightGBM | Fold 1 | AUC: 0.960956
LightGBM | Fold 2 | AUC: 0.961550
LightGBM | Fold 3 | AUC: 0.962062
LightGBM | Fold 4 | AUC: 0.962802
LightGBM | Fold 5 | AUC: 0.961694

LightGBM OOF ROC-AUC: 0.961811
Mean Fold AUC: 0.961813
Fold Standard Deviation: 0.000610


In [20]:
def build_xgboost(fold):

    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=5,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.5,
        gamma=0.05,
        random_state=(
            RANDOM_STATE + fold
        ),
        n_jobs=-1
    )


xgboost_result = train_with_cv(
    model_builder=build_xgboost,
    X_data=X_encoded,
    y_data=y,
    X_test_data=X_test_encoded,
    model_name="XGBoost"
)

XGBoost | Fold 1 | AUC: 0.956619
XGBoost | Fold 2 | AUC: 0.957246
XGBoost | Fold 3 | AUC: 0.957727
XGBoost | Fold 4 | AUC: 0.958666
XGBoost | Fold 5 | AUC: 0.957619

XGBoost OOF ROC-AUC: 0.957572
Mean Fold AUC: 0.957575
Fold Standard Deviation: 0.000669


In [21]:
catboost_oof_final = np.zeros(
    len(X_catboost)
)

catboost_test_final = np.zeros(
    len(X_test_catboost)
)

catboost_final_scores = []

for fold, (
    train_index,
    valid_index
) in enumerate(
    cv.split(
        X_catboost,
        y
    ),
    start=1
):

    X_train_fold = (
        X_catboost.iloc[
            train_index
        ]
    )

    X_valid_fold = (
        X_catboost.iloc[
            valid_index
        ]
    )

    y_train_fold = (
        y.iloc[
            train_index
        ]
    )

    y_valid_fold = (
        y.iloc[
            valid_index
        ]
    )

    model = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.02,
        depth=7,
        l2_leaf_reg=5.0,
        random_strength=0.5,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=(
            RANDOM_STATE + fold
        ),
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=(
            categorical_columns
        ),
        eval_set=(
            X_valid_fold,
            y_valid_fold
        ),
        early_stopping_rounds=200,
        verbose=False
    )

    valid_predictions = (
        model.predict_proba(
            X_valid_fold
        )[:, 1]
    )

    catboost_oof_final[
        valid_index
    ] = valid_predictions

    catboost_test_final += (
        model.predict_proba(
            X_test_catboost
        )[:, 1]
        / N_SPLITS
    )

    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_predictions
    )

    catboost_final_scores.append(
        fold_auc
    )

    print(
        f"CatBoost | "
        f"Fold {fold} | "
        f"AUC: {fold_auc:.6f}"
    )


catboost_final_auc = roc_auc_score(
    y,
    catboost_oof_final
)

print("\n" + "=" * 70)

print(
    "CatBoost Final "
    f"OOF ROC-AUC: "
    f"{catboost_final_auc:.6f}"
)

print(
    "Mean Fold AUC: "
    f"{np.mean(catboost_final_scores):.6f}"
)

print(
    "Fold Standard Deviation: "
    f"{np.std(catboost_final_scores):.6f}"
)



CatBoost | Fold 1 | AUC: 0.959166
CatBoost | Fold 2 | AUC: 0.959930
CatBoost | Fold 3 | AUC: 0.960218
CatBoost | Fold 4 | AUC: 0.961080
CatBoost | Fold 5 | AUC: 0.960114

CatBoost Final OOF ROC-AUC: 0.960100
Mean Fold AUC: 0.960102
Fold Standard Deviation: 0.000613


In [22]:
all_results = [
    {
        "Model": "Logistic Regression",
        "OOF_AUC": logistic_auc,
        "Mean_Fold_AUC": (
            np.mean(
                logistic_fold_scores
            )
        ),
        "Fold_STD": (
            np.std(
                logistic_fold_scores
            )
        )
    },
    {
        "Model": "Random Forest",
        "OOF_AUC": (
            random_forest_result[
                "auc"
            ]
        ),
        "Mean_Fold_AUC": (
            np.mean(
                random_forest_result[
                    "fold_scores"
                ]
            )
        ),
        "Fold_STD": (
            np.std(
                random_forest_result[
                    "fold_scores"
                ]
            )
        )
    },
    {
        "Model": (
            "HistGradientBoosting"
        ),
        "OOF_AUC": (
            hist_gradient_result[
                "auc"
            ]
        ),
        "Mean_Fold_AUC": (
            np.mean(
                hist_gradient_result[
                    "fold_scores"
                ]
            )
        ),
        "Fold_STD": (
            np.std(
                hist_gradient_result[
                    "fold_scores"
                ]
            )
        )
    },
    {
        "Model": "CatBoost",
        "OOF_AUC": (
            catboost_final_auc
        ),
        "Mean_Fold_AUC": (
            np.mean(
                catboost_final_scores
            )
        ),
        "Fold_STD": (
            np.std(
                catboost_final_scores
            )
        )
    },
    {
        "Model": "LightGBM",
        "OOF_AUC": (
            lightgbm_result[
                "auc"
            ]
        ),
        "Mean_Fold_AUC": (
            np.mean(
                lightgbm_result[
                    "fold_scores"
                ]
            )
        ),
        "Fold_STD": (
            np.std(
                lightgbm_result[
                    "fold_scores"
                ]
            )
        )
    },
    {
        "Model": "XGBoost",
        "OOF_AUC": (
            xgboost_result[
                "auc"
            ]
        ),
        "Mean_Fold_AUC": (
            np.mean(
                xgboost_result[
                    "fold_scores"
                ]
            )
        ),
        "Fold_STD": (
            np.std(
                xgboost_result[
                    "fold_scores"
                ]
            )
        )
    }
]

results_df = pd.DataFrame(
    all_results
)

results_df = (
    results_df
    .sort_values(
        "OOF_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    results_df.style.format(
        {
            "OOF_AUC": "{:.6f}",
            "Mean_Fold_AUC": "{:.6f}",
            "Fold_STD": "{:.6f}"
        }
    )
)

,Model,OOF_AUC,Mean_Fold_AUC,Fold_STD
0,LightGBM,0.961811,0.961813,0.000610
1,HistGradientBoosting,0.961537,0.961539,0.000564
2,CatBoost,0.960100,0.960102,0.000613
3,XGBoost,0.957572,0.957575,0.000669
4,Random Forest,0.942694,0.942699,0.000817
5,Logistic Regression,0.923249,0.923251,0.000571


In [23]:
model_predictions = {
    "logistic": {
        "oof": logistic_oof,
        "test": None,
        "auc": logistic_auc
    },

    "random_forest": {
        "oof": (
            random_forest_result[
                "oof"
            ]
        ),
        "test": (
            random_forest_result[
                "test"
            ]
        ),
        "auc": (
            random_forest_result[
                "auc"
            ]
        )
    },

    "hist_gradient": {
        "oof": (
            hist_gradient_result[
                "oof"
            ]
        ),
        "test": (
            hist_gradient_result[
                "test"
            ]
        ),
        "auc": (
            hist_gradient_result[
                "auc"
            ]
        )
    },

    "catboost": {
        "oof": (
            catboost_oof_final
        ),
        "test": (
            catboost_test_final
        ),
        "auc": (
            catboost_final_auc
        )
    },

    "lightgbm": {
        "oof": (
            lightgbm_result[
                "oof"
            ]
        ),
        "test": (
            lightgbm_result[
                "test"
            ]
        ),
        "auc": (
            lightgbm_result[
                "auc"
            ]
        )
    },

    "xgboost": {
        "oof": (
            xgboost_result[
                "oof"
            ]
        ),
        "test": (
            xgboost_result[
                "test"
            ]
        ),
        "auc": (
            xgboost_result[
                "auc"
            ]
        )
    }
}

print(
    "Predictions stored successfully!"
)

Predictions stored successfully!


In [24]:
from itertools import combinations

available_models = [
    "random_forest",
    "hist_gradient",
    "catboost",
    "lightgbm",
    "xgboost"
]

ensemble_results = []

for number_of_models in range(
    2,
    len(available_models) + 1
):

    for model_group in combinations(
        available_models,
        number_of_models
    ):

        ensemble_oof = np.mean(
            [
                model_predictions[
                    model_name
                ]["oof"]
                for model_name
                in model_group
            ],
            axis=0
        )

        ensemble_auc = roc_auc_score(
            y,
            ensemble_oof
        )

        ensemble_results.append(
            {
                "models": (
                    " + ".join(
                        model_group
                    )
                ),
                "number_of_models": (
                    number_of_models
                ),
                "OOF_AUC": (
                    ensemble_auc
                )
            }
        )

ensemble_results_df = (
    pd.DataFrame(
        ensemble_results
    )
    .sort_values(
        "OOF_AUC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    ensemble_results_df
    .head(15)
    .style.format(
        {
            "OOF_AUC": "{:.6f}"
        }
    )
)

,models,number_of_models,OOF_AUC
0,hist_gradient + lightgbm,2,0.961796
1,hist_gradient + catboost + lightgbm,3,0.961639
2,catboost + lightgbm,2,0.961439
3,hist_gradient + catboost,2,0.961318
4,hist_gradient + catboost + lightgbm + xgboost,4,0.960950
5,hist_gradient + lightgbm + xgboost,3,0.960865
6,catboost + lightgbm + xgboost,3,0.960535
7,hist_gradient + catboost + xgboost,3,0.960455
8,lightgbm + xgboost,2,0.960265
9,hist_gradient + xgboost,2,0.960135


In [25]:
best_ensemble = (
    ensemble_results_df
    .iloc[0]
)

best_model_names = (
    best_ensemble[
        "models"
    ]
    .split(" + ")
)

print(
    "Best ensemble:"
)

print(
    best_ensemble[
        "models"
    ]
)

print(
    "Best OOF AUC:",
    f"{best_ensemble['OOF_AUC']:.6f}"
)

final_oof_predictions = np.mean(
    [
        model_predictions[
            model_name
        ]["oof"]
        for model_name
        in best_model_names
    ],
    axis=0
)

final_test_predictions = np.mean(
    [
        model_predictions[
            model_name
        ]["test"]
        for model_name
        in best_model_names
    ],
    axis=0
)

final_ensemble_auc = roc_auc_score(
    y,
    final_oof_predictions
)

print(
    "\nFinal ensemble OOF AUC:",
    f"{final_ensemble_auc:.6f}"
)

Best ensemble:
hist_gradient + lightgbm
Best OOF AUC: 0.961796

Final ensemble OOF AUC: 0.961796


In [26]:
submission = pd.DataFrame(
    {
        "id": test_df["id"],
        "addicted_label": (
            final_test_predictions
        )
    }
)

# Safety check
submission["addicted_label"] = (
    submission[
        "addicted_label"
    ]
    .clip(
        0,
        1
    )
)

submission.to_csv(
    "submission.csv",
    index=False
)

print(
    "submission.csv created successfully!"
)

print(
    "Submission shape:",
    submission.shape
)

print(
    "\nFirst 10 rows:"
)

display(
    submission.head(10)
)

print(
    "\nPrediction summary:"
)

display(
    submission[
        "addicted_label"
    ]
    .describe()
)

submission.csv created successfully!
Submission shape: (296302, 2)

First 10 rows:


,id,addicted_label
0,691369,0.998565
1,691370,0.954153
2,691371,0.948223
3,691372,0.981982
4,691373,0.995903
5,691374,0.892169
6,691375,0.888916
7,691376,0.456928
8,691377,0.980909
9,691378,0.446074



Prediction summary:


count    296302.000000
mean          0.709404
std           0.358280
min           0.000847
25%           0.414459
50%           0.926345
75%           0.998625
max           0.999991
Name: addicted_label, dtype: float64

In [27]:
assert (
    submission.columns.tolist()
    == [
        "id",
        "addicted_label"
    ]
), "Incorrect submission columns!"

assert (
    len(submission)
    == len(test_df)
), "Submission row count is incorrect!"

assert (
    submission["id"]
    .equals(
        test_df["id"]
    )
), "IDs are not aligned!"

assert (
    submission[
        "addicted_label"
    ]
    .between(
        0,
        1
    )
    .all()
), "Predictions must be between 0 and 1!"

assert (
    submission[
        "addicted_label"
    ]
    .nunique()
    > 2
), (
    "Predictions appear to be "
    "hard labels instead of probabilities!"
)

print(
    "Submission validation passed!"
)

print(
    "\nFinal file:"
)

print(
    "submission.csv"
)

Submission validation passed!

Final file:
submission.csv


In [28]:
EXPECTED_ROWS = 296_302

# Required submission columns

EXPECTED_COLUMNS = [
"id",
"addicted_label"
]

# Check exact column names and order

assert (
submission.columns.tolist()
== EXPECTED_COLUMNS
), (
f"Incorrect columns! "
f"Expected: {EXPECTED_COLUMNS}, "
f"Found: {submission.columns.tolist()}"
)

# Check exact number of data rows

assert (
len(submission)
== EXPECTED_ROWS
), (
f"Incorrect number of rows! "
f"Expected: {EXPECTED_ROWS:,} rows, "
f"Found: {len(submission):,} rows"
)

# Check that the test dataset also has the expected size

assert (
len(test_df)
== EXPECTED_ROWS
), (
f"test.csv has an unexpected number of rows! "
f"Expected: {EXPECTED_ROWS:,}, "
f"Found: {len(test_df):,}"
)

# Check that IDs are aligned correctly

assert (
submission["id"]
.equals(test_df["id"])
), "Submission IDs are not aligned with test.csv!"

# Check for missing IDs

assert (
submission["id"]
.isna()
.sum()
== 0
), "The submission contains missing IDs!"

# Check for duplicate IDs

assert (
submission["id"]
.duplicated()
.sum()
== 0
), "The submission contains duplicate IDs!"

# Check for missing predictions

assert (
submission["addicted_label"]
.isna()
.sum()
== 0
), "The submission contains missing predictions!"

# Check that predictions are valid probabilities

assert (
submission["addicted_label"]
.between(0, 1)
.all()
), "Predictions must be between 0 and 1!"

# Check that predictions are probabilities,

# not only hard labels such as 0 and 1

assert (
submission["addicted_label"]
.nunique()
> 2
), (
"Predictions appear to be hard labels "
"instead of continuous probabilities!"
)

# Save with the header included

submission.to_csv(
"submission.csv",
index=False,
header=True
)

# Verify the saved file

saved_submission = pd.read_csv(
"submission.csv"
)

assert (
saved_submission.shape
== (
EXPECTED_ROWS,
2
)
), (
"The saved submission file has an "
"unexpected shape!"
)

assert (
saved_submission.columns.tolist()
== EXPECTED_COLUMNS
), (
"The saved submission file has "
"incorrect headers!"
)

print(" Submission validation passed!")
print(
f"Data rows: {len(submission):,}"
)
print(
f" Header: {', '.join(EXPECTED_COLUMNS)}"
)
print(
f" Columns: {len(submission.columns)}"
)
print(
" File created: submission.csv"
)

print("\nSubmission preview:")
display(
submission.head()
)


 Submission validation passed!
Data rows: 296,302
 Header: id, addicted_label
 Columns: 2
 File created: submission.csv

Submission preview:


,id,addicted_label
0,691369,0.998565
1,691370,0.954153
2,691371,0.948223
3,691372,0.981982
4,691373,0.995903
